# PPI-molde · Comparece MT
### Do dado ao MVP: previsão de abstenção e priorização de mobilização eleitoral em MT
**Aprendizagem Computacional · 4º TIA · versão INICIAL · v2 (09/11/2026)**

Este notebook é o **exemplo de referência** da PPI GovTech. Ele percorre o CRISP-DM inteiro **num único problema**, das fontes oficiais ao dashboard, e cada seção corresponde a uma entrega das sprints e a uma prática semanal, onde o mesmo pedaço é construído passo a passo.

| Seção | Fase CRISP-DM | Entrega da sprint | Prática |
|---|---|---|---|
| 1 | Entendimento do negócio | Check-in 0 (03/11): pergunta, decisor, X e y | 27/10 · O problema |
| 2–3 | Entendimento e preparação dos dados | Check-in 1 (10/11): base analítica v0 + EDA | 03/11 · O ETL |
| 4 | Modelagem e avaliação (baselines) | Review 1 (17/11): baseline × modelo | 10/11 · Experimento honesto |
| 5 | Comparação de modelos | Check-in 3 (24/11): modelo escolhido | 17/11 · Validação cruzada e árvores |
| 6–8 | Perfis + da previsão à decisão | Check-in 3 (24/11): decisão v0 | 24/11 · Perfis e decisão |
| 9 | Implantação: MVP | Review 2 (01/12): previsões, ranking e dashboard | 01/12 · Tribunal do Pipeline |
| 10 | Checklist e limitações | Artigo SBC e banca | |

**Como usar:** rode de cima para baixo no Google Colab (*Ambiente de execução → Executar tudo*). A base fica em `dados/base_analitica.csv` e é gerada pelo ETL em `etl/etl_comparece_mt.py`. Ao final, a pasta de saída (impressa na 1ª célula de código) terá `previsoes.csv`, `ranking_decisao.csv` e `dashboard_comparece_mt.html`.

> **O que mudou na v2:** a v1 usava só 4 variáveis de perfil e errava quase tanto quanto prever a média (R² de 0,09 no teste). A v2 acrescenta a **abstenção de 2022**, o **crescimento do eleitorado** e a **distância até Cuiabá**, todas conhecidas antes da eleição, e compara o modelo com um baseline mais exigente: repetir a eleição anterior.

> **Aviso:** custos e efeito da ação de mobilização (seção 8) são **hipóteses didáticas**, não dados oficiais. O grupo de vocês deve buscar parâmetros reais ou declarar as hipóteses da mesma forma.

**Como usar a versão INICIAL:** há 6 lacunas `# TODO`, cada uma ligada à prática da semana em que aquele pedaço foi construído. Complete e rode de cima para baixo. Se travar, abra a prática correspondente.

## 1 · Entendimento do negócio  ·  *prática de 27/10 · apostila: cap. 1 a 3*
*Entrega do check-in 0 (03/11). Antes de abrir os dados, escrevam isto em uma frase por item.*

| Item | No PPI-molde |
|---|---|
| **Problema público** | Em MT, mais de 1 em cada 5 eleitores aptos (22%) não votou no 1º turno de 2026. Abstenção alta enfraquece a representatividade. |
| **Decisor** | Comitê de mobilização (TRE-MT em parceria com prefeituras), que tem um orçamento limitado para ações de transporte e comunicação no dia da eleição. |
| **Decisão apoiada** | Em **quais municípios** investir a ação, dado um orçamento. |
| **Pergunta analítica** | Com o que se sabe **antes** da eleição (abstenção anterior, perfil do eleitorado e geografia), dá para prever a abstenção de cada município melhor do que simplesmente repetir a eleição anterior? |
| **Unidade de análise** | Município (141 linhas). |
| **y (alvo)** | `ABSTENCAO_PCT`: % de aptos que não votaram no 1º turno para presidente em 2026. |
| **X (entradas)** | Abstenção em 2022; crescimento do eleitorado de 2022 a 2026; distância da sede até Cuiabá; % de 16 a 24 anos; % de 65 anos ou mais; % com superior completo; log do tamanho do eleitorado. |
| **Tarefa** | Regressão supervisionada (y numérico) + otimização (escolha sob orçamento). |

**Hipóteses testáveis**
- **H1 (hábito):** quem tinha abstenção alta em 2022 tende a ter abstenção alta em 2026.
- **H2 (distância):** municípios mais longe da capital têm abstenção maior.
- **H3 (mudança):** municípios cujo eleitorado cresceu muito (migração recente) têm abstenção maior que em 2022.
- **H4 (valor do modelo):** o modelo erra menos que os dois baselines: prever a média e repetir 2022.

**Pergunta para discussão:** por que a decisão vem antes do modelo?

**Resposta esperada:** porque é a decisão que define o que precisa ser previsto, com que unidade e com que métrica. Sem decisão, o grupo escolhe o modelo pelo que é "legal", e não pelo que ajuda o gestor. No molde, a decisão "onde investir" pede uma previsão **por município**, feita **antes** da eleição, numa métrica em **pontos percentuais**, que o gestor entende. É por isso que só entram em X variáveis conhecidas antes do dia da votação.

## 2 · Entendimento dos dados  ·  *prática de 03/11 · apostila: cap. 3, 4 e 6*
*Entrega do check-in 1 (10/11): base analítica v0, dicionário e EDA mínima.*

**Fontes:**
- TSE, perfil do eleitorado 2026 (geração 14/07/2026): `perfil_eleitorado_2026.zip`, em <https://dadosabertos.tse.jus.br/dataset/eleitorado-2026>.
- TSE, resultados por município e zona, 1º turno de 2026 (04/10/2026) e de 2022 (02/10/2022): `detalhe_votacao_munzona_2026.zip` e `detalhe_votacao_munzona_2022.zip`, em <https://dadosabertos.tse.jus.br>.
- Coordenadas das sedes municipais (IBGE), pelo repositório público `kelvins/municipios-brasileiros` no GitHub.

**ETL em código** (`etl/etl_comparece_mt.py`, construído na prática de 03/11):
1. **Extrair:** ler, de dentro dos três ZIP do TSE, só o CSV de MT e só as colunas necessárias (`sep=';'`, `encoding='latin1'`). O ZIP é lido direto do servidor, aos pedaços, sem baixar os 400 MB.
2. **Transformar:** somar o perfil por município e calcular os percentuais; filtrar os resultados para presidente (cargo 1), 1º turno, MT; somar aptos e abstenções das zonas; calcular a abstenção de 2026 (y) e de 2022 (X); calcular o crescimento do eleitorado e a distância até Cuiabá.
3. **Validar:** chave única, percentuais entre 0 e 100, nenhum município sem coordenada.
4. **Carregar:** salvar `dados/base_analitica.csv`.

Um município, **Boa Esperança do Norte**, não aparece nos resultados de 2022 (foi instalado depois) e, por isso, não tem abstenção anterior. O ETL o retira da base e avisa. Essa é uma decisão de projeto e precisa estar escrita no artigo.

Para refazer a base a partir das fontes, mude `REFAZER_BASE` para `True` na célula abaixo. Os ZIP do TSE são grandes (o do perfil tem cerca de 400 MB), mas o ETL lê só o trecho de MT de cada um, direto do servidor: são poucos MB e cerca de um minuto.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
SEMENTE = 42
REFAZER_BASE = False   # True: refaz a base a partir das fontes oficiais (lê só o trecho de MT dos ZIP do TSE)
# Arquivos gerados ficam FORA da pasta do projeto: /content no Colab, sua pasta de usuário no laboratório
SAIDA = (Path('/content') if Path('/content').exists() else Path.home() / 'ac_saida') / 'comparece_mt'
SAIDA.mkdir(parents=True, exist_ok=True)
print('Arquivos gerados vão para:', SAIDA)

REPO = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/ppi_molde_comparece_mt/'
ARQUIVO = 'base_analitica.csv'

if REFAZER_BASE:
    import sys, urllib.request
    etl = next((p for p in [Path('etl'), Path('..') / 'etl'] if (p / 'etl_comparece_mt.py').exists()), None)
    if etl is None:   # no Colab, aberto pelo GitHub: baixa o script do repositório
        etl = SAIDA
        urllib.request.urlretrieve(REPO + 'etl/etl_comparece_mt.py', etl / 'etl_comparece_mt.py')
    sys.path.insert(0, str(etl))
    import etl_comparece_mt
    base = etl_comparece_mt.executar(saida=SAIDA / ARQUIVO)
else:
    # Procura a base na pasta dados/ do projeto; no Colab (aberto pelo GitHub), baixa do repositório
    candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
    local = next((p for p in candidatos if p.exists()), None)
    base = pd.read_csv(local if local is not None else REPO + 'dados/' + ARQUIVO)
    print('Base lida de:', local.resolve() if local is not None else REPO + 'dados/' + ARQUIVO)
base['CD_MUNICIPIO'] = base['CD_MUNICIPIO'].astype(str)
print(base.shape)
base.head()

### Checagens de qualidade (validação)
Cada `assert` é uma regra do negócio escrita em código. Se uma falhar, o notebook para e mostra onde está o problema. Isso é melhor do que descobrir na banca.

In [ ]:
assert base['CD_MUNICIPIO'].is_unique, 'município repetido'
assert base.isna().sum().sum() == 0, 'há valores ausentes'
assert base['ABSTENCAO_PCT'].between(0, 100).all() and base['ABST_2022_PCT'].between(0, 100).all()
assert (base['QT_ABSTENCOES'] <= base['QT_APTOS']).all()
recalculado = 100 * base['QT_ABSTENCOES'] / base['QT_APTOS']
assert np.allclose(recalculado, base['ABSTENCAO_PCT']), 'alvo inconsistente com as contagens'
assert base['DIST_CUIABA_KM'].between(0, 1500).all(), 'distância fora de MT'
print('Todas as checagens passaram:', len(base), 'municípios')

### Dicionário mínimo

| Variável | Descrição | Papel |
|---|---|---|
| `CD_MUNICIPIO`, `NM_MUNICIPIO` | Código TSE e nome | **ID**: nunca entra em X |
| `ABST_2022_PCT` | % de aptos que não votaram no 1º turno de 2022 | X (conhecida 4 anos antes) |
| `CRESC_ELEITORADO_PCT` | Variação % do eleitorado: perfil de julho/2026 contra aptos de 2022 | X (conhecida em julho) |
| `DIST_CUIABA_KM` | Distância em linha reta da sede até Cuiabá | X (fixa) |
| `PCT_16_24` | % do eleitorado com 16 a 24 anos | X |
| `PCT_65_MAIS` | % com 65 anos ou mais | X |
| `PCT_SUPERIOR_COMPLETO` | % com superior completo | X |
| `LOG_ELEITORADO` | log(1 + eleitores do perfil): tamanho do município em escala comprimida | X |
| `ELEITORADO_PERFIL`, `LATITUDE`, `LONGITUDE` | Total de eleitores em julho e posição da sede | Contexto (já representados em outras colunas) |
| `QT_APTOS`, `QT_ABSTENCOES` | Contagens do resultado de 2026 | **Proibidas em X**: formam o alvo (vazamento) |
| `ABSTENCAO_PCT` | 100 × abstenções ÷ aptos, em 2026 | **y** |

In [ ]:
ID = ['CD_MUNICIPIO', 'NM_MUNICIPIO']
X_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
          'ABST_2022_PCT', 'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
Y_COL = None  # TODO 1 (prática de 27/10): qual coluna é o alvo y?
base[X_COLS + [Y_COL]].describe().T

### EDA mínima: o alvo e suas relações
Três perguntas antes de modelar: como y se distribui? Há municípios fora do padrão? Cada X parece se relacionar com y?

In [ ]:
fig, eixos = plt.subplots(2, 4, figsize=(20, 7.5))
eixos = eixos.ravel()
eixos[0].hist(base[Y_COL], bins=20, color='#1f6feb', alpha=0.8)
eixos[0].set_title('Distribuição de y'); eixos[0].set_xlabel('Abstenção 2026 (%)')
for ax, col in zip(eixos[1:], X_COLS):
    ax.scatter(base[col], base[Y_COL], s=14, alpha=0.6)
    r = base[col].corr(base[Y_COL])
    ax.set_title(f'{col}  (r = {r:.2f})'); ax.set_xlabel(col); ax.set_ylabel('Abstenção 2026 (%)')
plt.tight_layout(); plt.show()

print('Municípios com abstenção mais alta em 2026:')
print(base.nlargest(5, Y_COL)[['NM_MUNICIPIO', Y_COL, 'ABST_2022_PCT', 'DIST_CUIABA_KM']].to_string(index=False))

**Leitura:** a abstenção de 2022 é, de longe, a variável mais associada a y (r ≈ 0,77): municípios que faltavam muito em 2022 continuam faltando em 2026 (H1). A distância até Cuiabá vem em seguida (r ≈ 0,55, H2), e o crescimento do eleitorado também se associa a mais abstenção (r ≈ 0,31, H3).

**Pergunta para discussão:** a correlação de +0,42 entre `PCT_16_24` e abstenção prova que jovens votam menos?

**Resposta esperada:** não. É uma associação **entre municípios**, não entre pessoas (cuidado com a falácia ecológica). Municípios com mais jovens podem diferir também em migração, distância das seções e renda. A correlação justifica testar a variável no modelo, mas não sustenta uma afirmação causal no artigo.

## 3 · Preparação: separar antes de aprender  ·  *prática de 10/11 · apostila: cap. 5 e 6*
O teste é separado **antes de qualquer ajuste** (escala, escolha de modelo, hiperparâmetro). Ele fica guardado e é usado **uma única vez**, no fim. Para comparar modelos, usaremos validação cruzada só dentro do treino.

Pense num inventário florestal: para saber se a equação de volume funciona, você a confere em parcelas que **não** usou para ajustá-la.

In [ ]:
from sklearn.model_selection import train_test_split

X = base[X_COLS]
y = base[Y_COL]
# TODO 2 (prática de 10/11): separe 20% para teste com random_state=SEMENTE
X_treino, X_teste, y_treino, y_teste = ...
assert not set(X_treino.index) & set(X_teste.index)
print('Treino:', len(X_treino), '| Teste (guardado):', len(X_teste))

## 4 · Modelagem: os baselines primeiro  ·  *prática de 10/11 · apostila: cap. 5 e 8*
*Entrega da review 1 (17/11).*

O **baseline** é a régua mínima. Aqui há **duas** réguas, e o modelo precisa passar das duas:
1. **Média:** prever sempre a média de y do treino (`DummyRegressor`). É a régua de quem não sabe nada.
2. **Persistência:** repetir a abstenção de 2022, somando a variação média entre as eleições vista no treino. É a régua de quem conhece o histórico, e é o que um gestor faria sem modelo.

A persistência não existe pronta no scikit-learn, então escrevemos uma classe pequena com `fit` e `predict`. Assim ela entra no mesmo protocolo que os outros modelos.

**Métrica: MAE (erro absoluto médio)**, em pontos percentuais. Escolhemos o MAE porque o gestor entende "o modelo erra, em média, 1,3 p.p.", e porque ele não é dominado por poucos municípios extremos, como acontece com o RMSE.

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

class Persistencia(BaseEstimator, RegressorMixin):
    """Baseline "a abstenção se repete": abstenção de 2022 + a variação média vista no treino."""
    def __init__(self, coluna='ABST_2022_PCT'):
        self.coluna = coluna
    def fit(self, X, y):
        self.desloc_ = float(np.mean(np.asarray(y) - X[self.coluna].to_numpy()))
        return self
    def predict(self, X):
        return X[self.coluna].to_numpy() + self.desloc_

cv = KFold(n_splits=5, shuffle=True, random_state=SEMENTE)

def avaliar_cv(modelo, nome):
    r = cross_validate(modelo, X_treino, y_treino, cv=cv, scoring='neg_mean_absolute_error')
    return {'modelo': nome, 'MAE_cv': -r['test_score'].mean(), 'desvio_cv': r['test_score'].std()}

baseline = None  # TODO 3 (prática de 10/11): DummyRegressor que prevê sempre a média
persistencia = Persistencia()
linear = None    # TODO 3: Pipeline com StandardScaler e LinearRegression
pd.DataFrame([avaliar_cv(baseline, 'Baseline (média)'), avaliar_cv(persistencia, 'Baseline (persistência)'),
              avaliar_cv(linear, 'Regressão linear')])

**Leitura:** só repetir 2022 já tira cerca de 30% do erro da média. A regressão linear, com as 7 variáveis, tira mais um quarto do erro da persistência, e o desvio entre dobras é pequeno: o ganho não depende do sorteio.

**Pergunta para discussão:** por que a `StandardScaler` está dentro do `Pipeline` e não aplicada antes, na base toda?

**Resposta esperada:** se a média e o desvio forem calculados na base toda, a informação das dobras de validação (e do teste) vaza para o ajuste. Dentro do `Pipeline`, a escala é recalculada só com o treino de cada dobra.

## 5 · Comparação justa de modelos  ·  *prática de 17/11 · apostila: cap. 10, 11, 13 e 16*
*Entrega do check-in 3 (24/11).* Mesmo protocolo (mesmas 5 dobras e mesma métrica), mudando **uma coisa por vez**: o algoritmo. Os hiperparâmetros (o k do k-NN e a profundidade da árvore) são escolhidos por `GridSearchCV`, também só no treino.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

busca_knn = GridSearchCV(
    Pipeline([('escala', StandardScaler()), ('modelo', KNeighborsRegressor())]),
    {'modelo__n_neighbors': [3, 5, 10, 15, 20, 30]},
    cv=cv, scoring='neg_mean_absolute_error').fit(X_treino, y_treino)
k_escolhido = busca_knn.best_params_['modelo__n_neighbors']
busca_arvore = GridSearchCV(DecisionTreeRegressor(random_state=SEMENTE), {'max_depth': [1, 2, 3, 4, 5, 6, 8, None]},
                            cv=cv, scoring='neg_mean_absolute_error').fit(X_treino, y_treino)
prof_escolhida = busca_arvore.best_params_['max_depth']
print('k escolhido:', k_escolhido, '| profundidade escolhida:', prof_escolhida)

candidatos = {
    'Baseline (média)': DummyRegressor(strategy='mean'),
    'Baseline (persistência)': Persistencia(),
    'Regressão linear': Pipeline([('escala', StandardScaler()), ('modelo', LinearRegression())]),
    'Ridge (alpha=1)': Pipeline([('escala', StandardScaler()), ('modelo', Ridge(alpha=1.0))]),
    f'k-NN (k={k_escolhido})': Pipeline([('escala', StandardScaler()), ('modelo', KNeighborsRegressor(n_neighbors=k_escolhido))]),
    f'Árvore (prof. {prof_escolhida})': DecisionTreeRegressor(max_depth=prof_escolhida, random_state=SEMENTE),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=SEMENTE),
}
comparacao = pd.DataFrame([avaliar_cv(m, n) for n, m in candidatos.items()]).sort_values('MAE_cv')
comparacao

**Decisão de modelagem:** escolhemos a **regressão linear**. Ela empata com a Ridge (diferença bem menor que o desvio entre dobras) e vence com folga k-NN, árvore e Random Forest. Com 112 municípios de treino e uma relação quase linear entre a abstenção de 2022 e a de 2026, os modelos flexíveis não têm dados suficientes para mostrar vantagem. Além disso, a linear é interpretável: cada coeficiente pode ser explicado ao gestor.

Só agora, com a escolha fixada, o teste é aberto **uma única vez**.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# TODO 4 (prática de 17/11): ajuste a regressão linear em TODO o treino (é o modelo escolhido)
modelo_final = ...
linhas = []
for nome in ['Baseline (média)', 'Baseline (persistência)', 'Regressão linear']:
    previsto = candidatos[nome].fit(X_treino, y_treino).predict(X_teste)
    linhas.append({'modelo': nome, 'MAE_teste': mean_absolute_error(y_teste, previsto),
                   'R2_teste': r2_score(y_teste, previsto)})
resultado_teste = pd.DataFrame(linhas)
MAE_MEDIA, MAE_BASE, MAE_MODELO = resultado_teste['MAE_teste']
print(f'Redução do erro em relação à média: {100 * (1 - MAE_MODELO / MAE_MEDIA):.0f}%'
      f' | em relação à persistência: {100 * (1 - MAE_MODELO / MAE_BASE):.0f}%')
resultado_teste

In [ ]:
coef = pd.Series(modelo_final.named_steps['modelo'].coef_, index=X_COLS).sort_values()
ax = coef.plot.barh(color=['#bf3989' if v < 0 else '#1f6feb' for v in coef], figsize=(7, 3.4))
ax.set_title('Efeito de +1 desvio-padrão em cada variável (p.p. de abstenção)')
ax.axvline(0, color='grey', lw=0.8); plt.tight_layout(); plt.show()
coef.round(2)

**Interpretação (para o artigo):**
- **H1 apoiada (hábito):** a abstenção de 2022 é o maior coeficiente. Município que faltava muito em 2022 continua faltando.
- **H3 apoiada (mudança):** onde o eleitorado cresceu mais desde 2022, a abstenção sobe além do que o histórico explicaria. Uma leitura possível: eleitores recém-chegados, que ainda votam longe ou transferem o título tarde. Isso é hipótese para o artigo, não conclusão.
- **H2 com ressalva (distância):** sozinha, a distância tem r ≈ 0,55 com y, mas no modelo o coeficiente dela é quase zero. A distância **explica por que** a abstenção de 2022 já era alta; com 2022 no modelo, ela não acrescenta informação nova. Isso não quer dizer que a distância "não importa".
- Depois de considerar o histórico, municípios maiores e mais envelhecidos faltam um pouco mais; os mais escolarizados, um pouco menos.
- **H4 apoiada:** no teste, o modelo erra 1,23 p.p., contra 1,96 da persistência e 2,13 da média, com R² de 0,77.

**Limitação honesta:** 29 municípios de teste é pouco, e outra semente daria números um pouco diferentes. Por isso reportamos também a validação cruzada, com média e desvio entre dobras. E a relação foi aprendida com **um** par de eleições (2022 → 2026); para a próxima, o ideal é validar também com 2018 → 2022.

## 6 · Complemento não supervisionado: perfis de municípios  ·  *prática de 24/11 · apostila: cap. 18 e 19*
O agrupamento **não usa y** nem a abstenção de 2022. Ele descreve o território com as variáveis de perfil e de geografia: que "tipos" de município existem? No dashboard, isso ajuda o gestor a entender a carteira escolhida. A escolha de k usa o **coeficiente de silhueta** (quanto mais perto de 1, mais separados os grupos).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PERFIL_COLS = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO',
               'CRESC_ELEITORADO_PCT', 'DIST_CUIABA_KM']
Z = StandardScaler().fit_transform(base[PERFIL_COLS])
silhuetas = {k: silhouette_score(Z, KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit_predict(Z)) for k in range(2, 7)}
print({k: round(v, 3) for k, v in silhuetas.items()})
K = max(silhuetas, key=silhuetas.get)
base['PERFIL'] = KMeans(n_clusters=K, n_init=10, random_state=SEMENTE).fit_predict(Z)
resumo_perfis = base.groupby('PERFIL').agg(
    n=('NM_MUNICIPIO', 'size'), pct_16_24=('PCT_16_24', 'mean'), pct_65=('PCT_65_MAIS', 'mean'),
    pct_sup=('PCT_SUPERIOR_COMPLETO', 'mean'), eleitorado_mediano=('ELEITORADO_PERFIL', 'median'),
    cresc=('CRESC_ELEITORADO_PCT', 'mean'), dist_km=('DIST_CUIABA_KM', 'mean'), abst_media=(Y_COL, 'mean'))
resumo_perfis

A silhueta é maior com k = 3, mas os valores são baixos (perto de 0,24): os grupos existem, mas se sobrepõem. É comum em dados de municípios, que variam de forma contínua. No artigo, apresente os perfis como **uma descrição útil**, e não como tipos naturais.

In [ ]:
# Nomes dados pela leitura da tabela acima (passo humano, não automático).
def nomear(linha):
    if linha['eleitorado_mediano'] == resumo_perfis['eleitorado_mediano'].max():
        return 'Polos maiores e mais escolarizados'
    if linha['pct_65'] == resumo_perfis['pct_65'].max():
        return 'Pequenos, envelhecidos e perto da capital'
    return 'Longe da capital e em expansão'
nomes_perfis = {i: nomear(l) for i, l in resumo_perfis.iterrows()}
base['PERFIL_NOME'] = base['PERFIL'].map(nomes_perfis)
resumo_perfis.assign(nome=resumo_perfis.index.map(nomes_perfis))

**Pergunta para discussão:** o grupo "Longe da capital e em expansão" tem a maior abstenção média. Isso significa que o K-Means "previu" abstenção?

**Resposta esperada:** não. O K-Means nunca viu a abstenção; ela foi calculada **depois**, por grupo, só para descrever. O que o agrupamento mostra é que distância e crescimento andam juntos no território, e que esse tipo de município é onde a abstenção se concentra. Os nomes dos perfis são interpretação humana e devem ser apresentados assim no artigo.

## 7 · Previsões para todos os municípios (sem otimismo)  ·  *prática de 24/11 · apostila: cap. 24*
Para decidir, precisamos de uma previsão para **cada** município. Se o modelo prever municípios que ele usou no ajuste, o erro parece menor do que é. Usamos `cross_val_predict`: cada município é previsto por um modelo que **não o viu** no treino.

In [ ]:
from sklearn.model_selection import cross_val_predict

# TODO 5 (prática de 24/11): previsões FORA DA AMOSTRA para todos os municípios com cross_val_predict
base['ABST_PREVISTA'] = ...
base['RESIDUO'] = base[Y_COL] - base['ABST_PREVISTA']
print(f"MAE fora da amostra ({len(base)} municípios): {mean_absolute_error(base[Y_COL], base['ABST_PREVISTA']):.2f} p.p.")
print('Maior abstenção acima do previsto (investigar):')
print(base.nlargest(5, 'RESIDUO')[['NM_MUNICIPIO', Y_COL, 'ABST_PREVISTA', 'RESIDUO']].to_string(index=False))

## 8 · Da previsão à decisão: otimização sob orçamento  ·  *apostila: cap. 24*
*Entrega do check-in 3 (24/11), prática de 24/11. Aqui o PPI encontra a disciplina de Otimização.*

**Hipóteses didáticas (declaradas, não oficiais):**
- **Meta:** abstenção de até 20%. A ação mira só o **excesso** previsto acima da meta.
- **Efeito:** a ação recupera 30% dos eleitores em excesso.
- **Custo:** R$ 10 mil fixos (equipe e logística) + R$ 1,50 por eleitor apto (transporte e comunicação).

Para cada município *i*:
- benefício previsto: bᵢ = 0,30 × max(abst_previstaᵢ − 20, 0) / 100 × aptosᵢ  (eleitores a mais)
- custo: cᵢ = 10 + 0,0015 × aptosᵢ  (R$ mil)

**Problema da mochila 0/1:** escolher xᵢ ∈ {0, 1} para maximizar Σ bᵢ·xᵢ sujeito a Σ cᵢ·xᵢ ≤ B. Resolvemos por programação dinâmica, que dá a solução **ótima** e exata para custos inteiros.

In [ ]:
META, EFEITO, FIXO, POR_ELEITOR = 20.0, 0.30, 10, 0.0015
ORCAMENTO = 1000  # R$ mil

base['CUSTO'] = np.ceil(FIXO + POR_ELEITOR * base['QT_APTOS']).astype(int)
# TODO 6 (prática de 24/11): benefício = EFEITO × max(abst. prevista − META, 0) / 100 × aptos
base['BENEF_PREV'] = ...
# Só para avaliação retrospectiva: o que a mesma regra daria com a abstenção real.
base['BENEF_REAL'] = EFEITO * np.clip(base[Y_COL] - META, 0, None) / 100 * base['QT_APTOS']

def mochila(custos, beneficios, B):
    custos, beneficios = np.asarray(custos), np.asarray(beneficios)
    dp = np.zeros(B + 1); guarda = np.zeros((len(custos), B + 1), dtype=bool)
    for i, (c, b) in enumerate(zip(custos, beneficios)):
        if b <= 0 or c > B:
            continue
        candidato = dp[:B + 1 - c] + b
        melhora = candidato > dp[c:]
        novo = dp.copy(); novo[c:][melhora] = candidato[melhora]
        guarda[i, c:] = melhora; dp = novo
    escolhidos, w = [], B
    for i in range(len(custos) - 1, -1, -1):
        if guarda[i, w]:
            escolhidos.append(i); w -= custos[i]
    return sorted(escolhidos)

def guloso(ordem, B):
    escolhidos, total = [], 0
    for i in ordem:
        if base.loc[i, 'BENEF_PREV'] > 0 and total + base.loc[i, 'CUSTO'] <= B:
            escolhidos.append(i); total += base.loc[i, 'CUSTO']
    return escolhidos

estrategias = {
    'Ótima (mochila)': mochila(base['CUSTO'], base['BENEF_PREV'], ORCAMENTO),
    'Maior % prevista primeiro': guloso(base['ABST_PREVISTA'].sort_values(ascending=False).index, ORCAMENTO),
    'Maiores municípios primeiro': guloso(base['QT_APTOS'].sort_values(ascending=False).index, ORCAMENTO),
}
tabela_estrategias = pd.DataFrame([{
    'estratégia': n, 'municípios': len(s), 'custo (R$ mil)': base.loc[s, 'CUSTO'].sum(),
    'eleitores a mais (previsto)': base.loc[s, 'BENEF_PREV'].sum(),
    'eleitores a mais (retrospectivo)': base.loc[s, 'BENEF_REAL'].sum()} for n, s in estrategias.items()])
tabela_estrategias

**Leitura:** pela previsão, a mochila é a melhor estratégia, como esperado: ela é ótima **para os números que recebe**. A coluna retrospectiva aplica a mesma regra à abstenção que realmente ocorreu, e a mochila continua na frente: com R$ 1 milhão, cerca de 8,2 mil eleitores a mais, contra 7,8 mil de "maior % prevista primeiro" e 5,1 mil de "maiores municípios primeiro".

Na v1 do molde, com o modelo fraco, era o contrário: a mochila concentrava dinheiro em Várzea Grande, prevista com 23,8% de abstenção quando a real foi 19,6%, e perdia na checagem retrospectiva. Agora Várzea Grande é prevista com 19,1%, abaixo da meta, e fica de fora. **A otimização amplifica o erro da previsão**: ela concentra recursos onde o modelo prevê o maior excesso. Melhorar a previsão melhorou a decisão.

**Pergunta para discussão:** como tornar a decisão ainda mais robusta sem trocar o modelo?

**Resposta esperada:** (1) otimizar sobre um cenário conservador (previsão menos o MAE); (2) testar a carteira em várias sementes e ficar com os municípios escolhidos com frequência; (3) olhar os maiores resíduos (seção 7) antes de fechar a lista: um município muito acima do previsto pode ter uma causa local que o modelo não vê. Isso é uma excelente seção de discussão no artigo.

In [ ]:
orcamentos = list(range(200, 3001, 200))
curvas = {n: [] for n in estrategias}
for B in orcamentos:
    sel = {
        'Ótima (mochila)': mochila(base['CUSTO'], base['BENEF_PREV'], B),
        'Maior % prevista primeiro': guloso(base['ABST_PREVISTA'].sort_values(ascending=False).index, B),
        'Maiores municípios primeiro': guloso(base['QT_APTOS'].sort_values(ascending=False).index, B),
    }
    for n, s in sel.items():
        curvas[n].append(base.loc[s, 'BENEF_REAL'].sum())
fig, ax = plt.subplots(figsize=(8, 3.8))
for n, v in curvas.items():
    ax.plot(orcamentos, v, marker='o', ms=3, label=n)
ax.set_xlabel('Orçamento (R$ mil)'); ax.set_ylabel('Eleitores a mais (retrospectivo)')
ax.set_title('Checagem retrospectiva das estratégias'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9 · Implantação: o MVP  ·  *apostila: cap. 24*
*Entrega da review 2 (01/12), revisada no Tribunal do Pipeline (prática de 01/12).* O notebook é o **motor** e gera três arquivos que alimentam qualquer vitrine:

| Arquivo | Para quê |
|---|---|
| `saida/previsoes.csv` | Uma linha por município: previsão, resíduo, perfil, custo e benefício. **Fonte do Power BI.** |
| `saida/ranking_decisao.csv` | A carteira ótima com o orçamento padrão. |
| `saida/dashboard_comparece_mt.html` | Dashboard web autocontido, com controle de orçamento (abre no navegador, sem servidor). |

**Para quem vai de Power BI:** *Obter dados → Texto/CSV → `previsoes.csv`*. Crie (1) cartões de KPI, (2) uma tabela ordenada por `BENEF_PREV`, (3) um gráfico de dispersão `ABST_PREVISTA` × `ABSTENCAO_PCT`, (4) uma segmentação por `PERFIL_NOME`. A mochila continua no notebook, e o Power BI só mostra o resultado.

In [ ]:
colunas_saida = ID + X_COLS + ['QT_APTOS', Y_COL, 'ABST_PREVISTA', 'RESIDUO', 'PERFIL_NOME', 'CUSTO', 'BENEF_PREV', 'BENEF_REAL']
base[colunas_saida].to_csv(SAIDA / 'previsoes.csv', index=False, encoding='utf-8-sig')
carteira = base.loc[estrategias['Ótima (mochila)'], colunas_saida].sort_values('BENEF_PREV', ascending=False)
carteira.to_csv(SAIDA / 'ranking_decisao.csv', index=False, encoding='utf-8-sig')
print(f'Carteira ótima com R$ {ORCAMENTO} mil: {len(carteira)} municípios')
carteira[['NM_MUNICIPIO', 'QT_APTOS', 'ABST_PREVISTA', 'CUSTO', 'BENEF_PREV']].head(10)

### Dashboard web
O dashboard é um HTML com os dados em JSON e um pouco de JavaScript. A mochila é refeita no navegador quando o gestor muda o orçamento. Esse é o padrão mais simples de "web" para o MVP: nada de servidor, basta abrir o arquivo ou publicá-lo no GitHub Pages.

In [ ]:
import json

TEMPLATE_HTML = r'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Comparece MT · MVP de priorização</title>
<style>
  :root { --tinta:#1d2733; --suave:#5d6b7a; --linha:#dde3ea; --fundo:#f6f8fa; --acento:#1f6feb; --bom:#1a7f37; --alerta:#bf8700; }
  * { box-sizing:border-box; }
  body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; color:var(--tinta); background:var(--fundo); }
  header { background:#fff; border-bottom:1px solid var(--linha); padding:18px 28px; }
  header h1 { margin:0; font-size:20px; }
  header p { margin:4px 0 0; color:var(--suave); font-size:13px; }
  main { max-width:1180px; margin:0 auto; padding:20px 28px 40px; }
  .controles { background:#fff; border:1px solid var(--linha); border-radius:10px; padding:16px 20px; display:flex; gap:32px; flex-wrap:wrap; align-items:center; }
  .controles label { font-size:13px; color:var(--suave); display:block; margin-bottom:4px; }
  .controles input[type=range] { width:320px; }
  .controles select { font-size:14px; padding:4px 6px; }
  .valor { font-weight:600; font-size:16px; }
  .kpis { display:grid; grid-template-columns:repeat(4,1fr); gap:14px; margin:16px 0; }
  .kpi { background:#fff; border:1px solid var(--linha); border-radius:10px; padding:14px 16px; }
  .kpi .rotulo { font-size:12px; color:var(--suave); }
  .kpi .numero { font-size:26px; font-weight:650; margin-top:4px; }
  .kpi .nota { font-size:12px; color:var(--suave); margin-top:2px; }
  .grade { display:grid; grid-template-columns: 1.25fr 1fr; gap:14px; }
  .cartao { background:#fff; border:1px solid var(--linha); border-radius:10px; padding:16px 18px; }
  .cartao h2 { font-size:15px; margin:0 0 4px; }
  .cartao .sub { font-size:12px; color:var(--suave); margin:0 0 10px; }
  table { width:100%; border-collapse:collapse; font-size:13px; }
  th, td { text-align:left; padding:6px 8px; border-bottom:1px solid var(--linha); }
  th { color:var(--suave); font-weight:600; font-size:12px; }
  td.n, th.n { text-align:right; font-variant-numeric: tabular-nums; }
  .rolagem { max-height:420px; overflow:auto; }
  .comparacao td:first-child { font-weight:600; }
  .destaque { color:var(--acento); }
  footer { color:var(--suave); font-size:12px; margin-top:18px; line-height:1.5; }
  @media (max-width: 900px) { .kpis { grid-template-columns:repeat(2,1fr); } .grade { grid-template-columns:1fr; } }
</style>
</head>
<body>
<header>
  <h1>Comparece MT · onde investir em mobilização eleitoral</h1>
  <p>MVP da PPI-molde · AC 4º TIA · previsão de abstenção (regressão) + priorização sob orçamento (mochila 0/1). Custos e efeito da ação são hipóteses didáticas.</p>
</header>
<main>
  <section class="controles">
    <div>
      <label for="orcamento">Orçamento disponível</label>
      <input id="orcamento" type="range" min="100" max="3000" step="50" value="1000">
      <div class="valor" id="orcamentoTxt"></div>
    </div>
    <div>
      <label for="estrategia">Estratégia de escolha</label>
      <select id="estrategia">
        <option value="otimo">Ótima (mochila sobre a previsão)</option>
        <option value="pct">Maior abstenção prevista primeiro</option>
        <option value="maiores">Maiores municípios primeiro</option>
      </select>
    </div>
  </section>

  <section class="kpis">
    <div class="kpi"><div class="rotulo">Municípios atendidos</div><div class="numero" id="kMun"></div><div class="nota" id="kMunNota"></div></div>
    <div class="kpi"><div class="rotulo">Custo da carteira</div><div class="numero" id="kCusto"></div><div class="nota" id="kCustoNota"></div></div>
    <div class="kpi"><div class="rotulo">Eleitores a mais (previsto)</div><div class="numero destaque" id="kBen"></div><div class="nota">segundo a previsão do modelo</div></div>
    <div class="kpi"><div class="rotulo">Eleitores a mais (checagem retrospectiva)</div><div class="numero" id="kReal"></div><div class="nota">se usássemos a abstenção real do 1º turno</div></div>
  </section>

  <section class="grade">
    <div class="cartao">
      <h2>Carteira escolhida</h2>
      <p class="sub">Ordenada pelo benefício previsto. A linha mostra o custo e quantos eleitores a mais a ação deve trazer.</p>
      <div class="rolagem"><table id="tabela"><thead><tr><th>Município</th><th class="n">Aptos</th><th class="n">Abst. prevista</th><th class="n">Abst. real</th><th class="n">Custo (R$ mil)</th><th class="n">Eleitores a mais</th></tr></thead><tbody></tbody></table></div>
    </div>
    <div>
      <div class="cartao">
        <h2>As três estratégias com este orçamento</h2>
        <p class="sub">Mesmo orçamento e mesma hipótese de efeito; muda só a regra de escolha.</p>
        <table class="comparacao" id="comparacao"><thead><tr><th>Estratégia</th><th class="n">Mun.</th><th class="n">Previsto</th><th class="n">Retrospectivo</th></tr></thead><tbody></tbody></table>
      </div>
      <div class="cartao" style="margin-top:14px">
        <h2>O modelo erra quanto?</h2>
        <p class="sub" id="metricaTxt"></p>
        <svg id="dispersao" viewBox="0 0 420 300" width="100%" role="img" aria-label="Abstenção prevista versus real por município"></svg>
      </div>
    </div>
  </section>

  <section class="cartao" style="margin-top:14px">
    <h2>Perfis de municípios (agrupamento K-Means)</h2>
    <p class="sub">Grupos encontrados sem usar a abstenção; servem para descrever o território, não são classes verdadeiras.</p>
    <table id="perfis"><thead><tr><th>Perfil</th><th class="n">Municípios</th><th class="n">% 16–24</th><th class="n">% 65+</th><th class="n">% superior</th><th class="n">Eleitorado mediano</th><th class="n">Abst. real média</th></tr></thead><tbody></tbody></table>
  </section>

  <footer id="rodape"></footer>
</main>

<script>
const D = __DADOS__;
const fmt = (v, c=0) => v.toLocaleString('pt-BR', {minimumFractionDigits:c, maximumFractionDigits:c});
const mun = D.municipios;

function mochila(B) {
  // Programação dinâmica 0/1: custo inteiro em R$ mil, maximiza o benefício previsto.
  const n = mun.length;
  let dp = new Float64Array(B + 1);
  const guarda = [];
  for (let i = 0; i < n; i++) {
    const c = mun[i].custo, b = mun[i].benef_prev;
    const k = new Uint8Array(B + 1);
    if (b > 0 && c <= B) {
      const novo = dp.slice();
      for (let w = c; w <= B; w++) {
        const cand = dp[w - c] + b;
        if (cand > novo[w]) { novo[w] = cand; k[w] = 1; }
      }
      dp = novo;
    }
    guarda.push(k);
  }
  const sel = [];
  let w = B;
  for (let i = n - 1; i >= 0; i--) { if (guarda[i][w]) { sel.push(i); w -= mun[i].custo; } }
  return sel;
}
function guloso(chave, B) {
  const ordem = mun.map((m, i) => i).sort((a, b) => mun[b][chave] - mun[a][chave]);
  const sel = []; let t = 0;
  for (const i of ordem) { if (mun[i].benef_prev > 0 && t + mun[i].custo <= B) { sel.push(i); t += mun[i].custo; } }
  return sel;
}
const escolher = (e, B) => e === 'otimo' ? mochila(B) : e === 'pct' ? guloso('abst_prevista', B) : guloso('aptos', B);
const soma = (s, k) => s.reduce((a, i) => a + mun[i][k], 0);

function desenhar() {
  const B = +document.getElementById('orcamento').value;
  const e = document.getElementById('estrategia').value;
  document.getElementById('orcamentoTxt').textContent = 'R$ ' + fmt(B) + ' mil';
  const sel = escolher(e, B);
  document.getElementById('kMun').textContent = sel.length;
  document.getElementById('kMunNota').textContent = 'de ' + mun.length + ' municípios de MT';
  document.getElementById('kCusto').textContent = 'R$ ' + fmt(soma(sel, 'custo')) + ' mil';
  document.getElementById('kCustoNota').textContent = fmt(100 * soma(sel, 'custo') / B) + '% do orçamento';
  document.getElementById('kBen').textContent = fmt(soma(sel, 'benef_prev'));
  document.getElementById('kReal').textContent = fmt(soma(sel, 'benef_real'));
  const corpo = document.querySelector('#tabela tbody');
  corpo.innerHTML = sel.slice().sort((a, b) => mun[b].benef_prev - mun[a].benef_prev).map(i => {
    const m = mun[i];
    return `<tr><td>${m.nome}</td><td class="n">${fmt(m.aptos)}</td><td class="n">${fmt(m.abst_prevista,1)}%</td><td class="n">${fmt(m.abst_real,1)}%</td><td class="n">${fmt(m.custo)}</td><td class="n">${fmt(m.benef_prev)}</td></tr>`;
  }).join('');
  const nomes = {otimo:'Ótima (mochila)', pct:'Maior % prevista', maiores:'Maiores municípios'};
  document.querySelector('#comparacao tbody').innerHTML = ['otimo','pct','maiores'].map(k => {
    const s = escolher(k, B);
    return `<tr${k===e?' class="destaque"':''}><td>${nomes[k]}</td><td class="n">${s.length}</td><td class="n">${fmt(soma(s,'benef_prev'))}</td><td class="n">${fmt(soma(s,'benef_real'))}</td></tr>`;
  }).join('');
}

function dispersao() {
  const svg = document.getElementById('dispersao');
  const W = 420, H = 300, m = {e:46, d:14, t:12, b:40};
  const vals = mun.flatMap(x => [x.abst_prevista, x.abst_real]);
  const lo = Math.floor(Math.min(...vals)), hi = Math.ceil(Math.max(...vals));
  const X = v => m.e + (v - lo) / (hi - lo) * (W - m.e - m.d);
  const Y = v => H - m.b - (v - lo) / (hi - lo) * (H - m.t - m.b);
  let s = '';
  for (let v = Math.ceil(lo / 2) * 2; v <= hi; v += 2) {
    s += `<line x1="${X(v)}" x2="${X(v)}" y1="${m.t}" y2="${H-m.b}" stroke="#eef1f4"/><text x="${X(v)}" y="${H-m.b+16}" font-size="11" text-anchor="middle" fill="#5d6b7a">${v}</text>`;
    s += `<line x1="${m.e}" x2="${W-m.d}" y1="${Y(v)}" y2="${Y(v)}" stroke="#eef1f4"/><text x="${m.e-6}" y="${Y(v)+4}" font-size="11" text-anchor="end" fill="#5d6b7a">${v}</text>`;
  }
  s += `<line x1="${X(lo)}" y1="${Y(lo)}" x2="${X(hi)}" y2="${Y(hi)}" stroke="#9aa6b2" stroke-dasharray="4 4"/>`;
  s += mun.map(x => `<circle cx="${X(x.abst_prevista)}" cy="${Y(x.abst_real)}" r="${2.5 + Math.sqrt(x.aptos)/120}" fill="#1f6feb" fill-opacity="0.35" stroke="#1f6feb" stroke-width="0.6"><title>${x.nome}: prevista ${fmt(x.abst_prevista,1)}%, real ${fmt(x.abst_real,1)}%</title></circle>`).join('');
  s += `<text x="${(W+m.e)/2}" y="${H-6}" font-size="12" text-anchor="middle" fill="#1d2733">Abstenção prevista (%)</text>`;
  s += `<text x="12" y="${(H-m.b)/2}" font-size="12" text-anchor="middle" fill="#1d2733" transform="rotate(-90 12 ${(H-m.b)/2})">Abstenção real (%)</text>`;
  svg.innerHTML = s;
  const r = D.metricas;
  document.getElementById('metricaTxt').textContent = `Erro médio absoluto no teste: modelo ${fmt(r.mae_modelo,2)} p.p. contra ${fmt(r.mae_baseline,2)} p.p. do melhor baseline (repetir 2022, corrigido). Cada ponto é um município, previsto por um modelo que não o viu no ajuste; na diagonal tracejada, acerto exato.`;
}

function perfis() {
  document.querySelector('#perfis tbody').innerHTML = D.perfis.map(p => `<tr><td>${p.nome}</td><td class="n">${p.n}</td><td class="n">${fmt(p.pct_16_24,1)}</td><td class="n">${fmt(p.pct_65,1)}</td><td class="n">${fmt(p.pct_sup,1)}</td><td class="n">${fmt(p.eleitorado_mediano)}</td><td class="n">${fmt(p.abst_media,1)}%</td></tr>`).join('');
}

document.getElementById('rodape').innerHTML = D.rodape;
document.getElementById('orcamento').addEventListener('input', desenhar);
document.getElementById('estrategia').addEventListener('change', desenhar);
dispersao(); perfis(); desenhar();
</script>
</body>
</html>
'''

def nome_bonito(n):
    minusculas = {'De', 'Do', 'Da', 'Dos', 'Das', 'E'}
    return ' '.join(p.lower() if p in minusculas and i else p for i, p in enumerate(n.title().split()))

dados_dash = {
    'municipios': [
        {'nome': nome_bonito(r.NM_MUNICIPIO), 'aptos': int(r.QT_APTOS), 'abst_real': round(r.ABSTENCAO_PCT, 2),
          'abst_prevista': round(r.ABST_PREVISTA, 2), 'custo': int(r.CUSTO),
          'benef_prev': round(r.BENEF_PREV, 1), 'benef_real': round(r.BENEF_REAL, 1), 'perfil': r.PERFIL_NOME}
        for r in base.itertuples()],
    'metricas': {'mae_modelo': MAE_MODELO, 'mae_baseline': MAE_BASE},
    'perfis': [
        {'nome': nomes_perfis[i], 'n': int(l.n), 'pct_16_24': l.pct_16_24, 'pct_65': l.pct_65,
          'pct_sup': l.pct_sup, 'eleitorado_mediano': float(l.eleitorado_mediano), 'abst_media': l.abst_media}
        for i, l in resumo_perfis.iterrows()],
    'rodape': ('Fontes: TSE, perfil do eleitorado 2026 (geração 14/07/2026) e resultados do 1º turno de 2022 e de 04/10/2026, '
               'via dadosabertos.tse.jus.br; coordenadas das sedes (IBGE). Hipóteses da ação: meta de 20%, efeito de 30% sobre o excesso, '
               'custo de R$ 10 mil + R$ 1,50 por eleitor apto. Material didático; não é recomendação oficial.'),
}
html = TEMPLATE_HTML.replace('__DADOS__', json.dumps(dados_dash, ensure_ascii=False))
(SAIDA / 'dashboard_comparece_mt.html').write_text(html, encoding='utf-8')
print('Dashboard salvo em', SAIDA / 'dashboard_comparece_mt.html')

In [ ]:
# No Colab, baixe os arquivos gerados (ignorado fora do Colab).
try:
    from google.colab import files
    for nome in ['previsoes.csv', 'ranking_decisao.csv', 'dashboard_comparece_mt.html']:
        files.download(str(SAIDA / nome))
except ImportError:
    print('Fora do Colab: os arquivos estão na pasta', SAIDA.resolve())

## 10 · Checklist das 6 perguntas (vai para o artigo e para a banca)  ·  *prática de 01/12 · apostila: cap. 24*

| Pergunta | Resposta no PPI-molde |
|---|---|
| 1. Qual é o y? | `ABSTENCAO_PCT`, % de aptos que não votaram no 1º turno de 2026, por município. |
| 2. Qual é o baseline? | Dois: prever sempre a média de y do treino (`DummyRegressor`) e repetir a abstenção de 2022 corrigida pela variação média (persistência). |
| 3. Por que essa métrica? | MAE em p.p.: interpretável pelo gestor e robusto a extremos. |
| 4. Onde foi feito o split? | Antes de tudo, 80/20 com `random_state=42`; comparação por validação cruzada de 5 dobras só no treino; teste usado uma vez. |
| 5. Há vazamento? | Só entram em X variáveis conhecidas antes da eleição; `QT_ABSTENCOES` e `QT_APTOS` de 2026 fora de X; escala dentro do `Pipeline`; previsões para decisão fora da amostra (`cross_val_predict`). |
| 6. A conclusão tem evidência? | No teste, MAE de 1,23 p.p., contra 1,96 da persistência e 2,13 da média, com R² de 0,77. Na validação cruzada, 1,33 contra 1,78 e 2,56. Na checagem retrospectiva, a carteira ótima traz mais eleitores que as regras simples em quase todos os orçamentos. |
| 7. Roda do zero? | Sim: `etl/etl_comparece_mt.py` refaz a base a partir das fontes oficiais, e este notebook roda de cima para baixo no Colab. |

**Limitações para declarar:** um único par de eleições (2022 → 2026); 141 municípios (Boa Esperança do Norte fica de fora por não ter histórico); hipóteses de custo e efeito não calibradas; a checagem retrospectiva usa o mesmo turno que gerou y. **Próximos passos:** validar com 2018 → 2022, incluir variáveis do IBGE (renda, população rural) e a distância às seções eleitorais, e não só à sede.